Pipeline de construcción del dataset de bienestar
================================================
Fusiona dos datasets de Kaggle mediante clustering y refina
las etiquetas con un Knowledge Graph basado en evidencia.

Datasets requeridos (descargar de Kaggle y colocar en el mismo directorio):
  - mental_health.csv   → Mental Health and Lifestyle Habits 2019-2024
  - holistic_health.csv → Holistic Health & Lifestyle Score Dataset

Preprocesamiento:
  1A/1B  → Limpiar y unificar tipos de cada dataset por separado.

  2A/2B  → Construir la etiqueta de cada dataset (bienestar_mental / fisico).

  3A/3B  → Normalizar las features comunes para poder comparar.

  4      → Merge: clustering por separado + matching de perfiles similares.

  5      → Añadir features sintéticas (las que ningún dataset tiene).
  

Exploración de características de los datasets

In [9]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

In [10]:
dfA = pd.read_csv("/content/Mental_Health_Lifestyle_Dataset.csv")
dfB = pd.read_csv("/content/holistic_health_lifestyle_dataset.csv")

print("=== DATASET A ===")
print(dfA.dtypes)
print("\nStress Level valores únicos:", dfA['Stress Level'].unique()[:10])
print("Exercise Level valores únicos:", dfA['Exercise Level'].unique()[:10])
print("Sleep Hours tipo:", dfA['Sleep Hours'].dtype, "| muestra:", dfA['Sleep Hours'].head(3).values)
print("Social Interaction Score tipo:", dfA['Social Interaction Score'].dtype)
print("Happiness Score tipo:", dfA['Happiness Score'].dtype)

print("\n=== DATASET B ===")
print(dfB.dtypes)
print("\nStress_Level valores únicos:", dfB['Stress_Level'].unique()[:10])
print("Physical_Activity valores únicos:", dfB['Physical_Activity'].unique()[:10])
print("Overall_Health_Score tipo:", dfB['Overall_Health_Score'].dtype, "| muestra:", dfB['Overall_Health_Score'].head(3).values)
print("Nutrition_Score tipo:", dfB['Nutrition_Score'].dtype)

=== DATASET A ===
Country                         object
Age                              int64
Gender                          object
Exercise Level                  object
Diet Type                       object
Sleep Hours                    float64
Stress Level                    object
Mental Health Condition         object
Work Hours per Week              int64
Screen Time per Day (Hours)    float64
Social Interaction Score       float64
Happiness Score                float64
dtype: object

Stress Level valores únicos: ['Low' 'High' 'Moderate']
Exercise Level valores únicos: ['Low' 'Moderate' 'High']
Sleep Hours tipo: float64 | muestra: [6.3 4.9 7.2]
Social Interaction Score tipo: float64
Happiness Score tipo: float64

=== DATASET B ===
Physical_Activity       float64
Nutrition_Score         float64
Stress_Level            float64
Mindfulness             float64
Sleep_Hours             float64
Hydration               float64
BMI                     float64
Alcohol                 

Inclusión de perfiles extra para representatividad de usuarios

In [11]:
COLUMNAS = [
 'edad','sexo','altura','bmi','peso',
 'h_sueno','c_sueno','n_estres',
 'h_movil','t_pantalla','pasos',
 'i_ejercicio','m_ejercicio',
 'alim_enteros','alim_procesados',
 'g_proteina','g_fibra',
 'h_social','h_sol',
 'l_agua','n_alcohol','t_meditacion',
 'bienestar_fisico','bienestar_mental'
]

usuario_inactivo = [
    27,    # edad
    1,     # sexo: 1=hombre
    1.75,  # altura (metros)
    23.0,  # bmi
    70.3,  # peso
    7.0,   # h_sueno
    3.5,   # c_sueno  [1-5]
    4.0,   # n_estres [1-10]
    2.0,   # h_movil
    4.0,   # t_pantalla
    2000,  # pasos
    2,     # i_ejercicio int [1-5]
    15,    # m_ejercicio
    3.0,   # alim_enteros [1-10]
    5.6,   # alim_procesados [0-8]
    80,    # g_proteina
    20,    # g_fibra
    2.0,   # h_social
    1.0,   # h_sol
    2.0,   # l_agua
    0.2,   # n_alcohol
    0,     # t_meditacion (minutos)
    45.0,  # bienestar_fisico [0-100]
    65.0   # bienestar_mental[0-100]
    ]
usuario_activo = [
    33,
    0,
    1.70,
    21.0,
    60.7,
    8.0,
    4.0,
    4.0,
    2.0,
    4.0,
    10000,
    3,
    45,
    7.0,
    2.4,
    65,
    24,
    2.0,
    1.5,
    2.2,
    0.2,
    0,
    75.0,
    85.0
]

usuario_sedentario_digital = [
    24,
    1,
    1.78,
    25.5,
    81.0,
    6.0,
    3.0,
    6.0,
    4.0,
    7.0,
    2500,
    1,
    5,
    4.0,
    4.8,
    75,
    16,
    1.5,
    0.5,
    1.8,
    0.3,
    0,
    40.0,
    55.0
]

usuario_estresado = [
    38,
    1,
    1.80,
    26.0,
    84.2,
    6.5,
    2.8,
    8.0,
    3.0,
    6.0,
    4000,
    2,
    10,
    5.0,
    4.0,
    90,
    18,
    1.0,
    0.4,
    1.7,
    0.4,
    0,
    45.0,
    40.0
]

usuario_fitness = [
    31,
    1,
    1.76,
    22.0,
    68.1,
    7.5,
    3.8,
    4.0,
    1.5,
    3.0,
    12000,
    4,
    60,
    8.0,
    1.6,
    120,
    30,
    2.0,
    1.2,
    2.5,
    0.1,
    12,
    85.0,
    80.0
]

usuario_mindfulness = [
    35,
    0,
    1.65,
    22.0,
    59.9,
    8.0,
    4.3,
    3.0,
    1.5,
    3.0,
    7000,
    3,
    25,
    7.5,
    2.0,
    70,
    28,
    3.0,
    1.0,
    2.3,
    0.1,
    21,
    70.0,
    90.0
]

usuario_desbalanceado = [
    28,
    1,
    1.82,
    24.0,
    79.5,
    5.5,
    2.5,
    6.0,
    2.5,
    5.0,
    9000,
    4,
    45,
    3.5,
    5.2,
    110,
    15,
    2.0,
    0.7,
    1.6,
    0.4,
    0,
    60.0,
    55.0
]

usuarios = [
    usuario_inactivo,
    usuario_activo,
    usuario_sedentario_digital,
    usuario_estresado,
    usuario_fitness,
    usuario_mindfulness,
    usuario_desbalanceado
]

In [12]:
# SIGMA: desviación del ruido aditivo por columna
# Columnas con sigma=0 no se perturban (binarias, enteras fijas, o perfil estático)
SIGMA = {
    'edad'            : 0,      # fijo por persona
    'sexo'            : 0,      # binario — no tocar
    'altura'          : 0,      # fijo por persona
    'bmi'             : 0.3,
    'peso'            : 0.8,
    'h_sueno'         : 0.4,    # horas
    'c_sueno'         : 0.2,    # escala 1–5
    'n_estres'        : 0.5,    # escala 1–10
    'h_movil'         : 0.3,
    't_pantalla'      : 0.4,
    'pasos'           : 400,
    'i_ejercicio'     : 0,      # int 1–5 — no perturbar
    'm_ejercicio'     : 5,      # minutos
    'alim_enteros'    : 0.5,    # escala 1–10
    'alim_procesados' : 0.4,    # escala 0–8
    'g_proteina'      : 8,
    'g_fibra'         : 3,
    'h_social'        : 0.3,
    'h_sol'           : 0.2,
    'l_agua'          : 0.15,
    'n_alcohol'       : 0.1,
    't_meditacion'    : 2,      # minutos
    'bienestar_fisico': 4.0,    # escala 0–100
    'bienestar_mental': 4.0,
}

# Límites físicos para clipear el resultado tras añadir ruido
LIMITES = {
    'bmi'             : (16,  45),
    'peso'            : (40,  150),
    'h_sueno'         : (3,   12),
    'c_sueno'         : (1,   5),
    'n_estres'        : (1,   10),
    'h_movil'         : (0,   14),
    't_pantalla'      : (0,   16),
    'pasos'           : (500, 25000),
    'm_ejercicio'     : (0,   180),
    'alim_enteros'    : (1,   10),
    'alim_procesados' : (0,   8),
    'g_proteina'      : (20,  250),
    'g_fibra'         : (2,   60),
    'h_social'        : (0,   12),
    'h_sol'           : (0,   6),
    'l_agua'          : (0.3, 5),
    'n_alcohol'       : (0,   10),
    't_meditacion'    : (0,   90),
    'bienestar_fisico': (0,   100),
    'bienestar_mental': (0,   100),
}

rng_hist = np.random.default_rng(42)  # seed fijo → reproducible

historial = []
for usuario in usuarios:
    base = dict(zip(COLUMNAS, usuario))
    for _ in range(10):
        dia = {}
        for col in COLUMNAS:
            sigma = SIGMA[col]
            val = base[col] + (rng_hist.normal(0, sigma) if sigma > 0 else 0)
            if col in LIMITES:
                lo, hi = LIMITES[col]
                val = float(np.clip(val, lo, hi))
            if col in ('pasos', 'm_ejercicio', 'i_ejercicio', 'edad', 'sexo', 't_meditacion'):
                val = int(round(val))
            dia[col] = val
        historial.append(dia)

hist_df = pd.DataFrame(historial)
print(f"Historial: {hist_df.shape}")  # esperado: (140, 24)
print(hist_df[['h_sueno','n_estres','i_ejercicio',
               'bienestar_fisico','bienestar_mental']].describe().round(2))

Historial: (70, 24)
       h_sueno  n_estres  i_ejercicio  bienestar_fisico  bienestar_mental
count    70.00     70.00        70.00             70.00             70.00
mean      6.93      4.89         2.71             60.24             67.05
std       0.98      1.72         1.04             17.12             17.62
min       4.69      1.95         1.00             30.42             31.01
25%       6.09      3.55         2.00             44.64             54.02
50%       7.06      4.42         3.00             60.68             65.10
75%       7.70      5.97         4.00             74.26             82.83
max       8.47      8.52         4.00             92.35             95.74


In [13]:
SEED = 42
rng  = np.random.default_rng(SEED)


# ─────────────────────────────────────────────────────────────
# UTILIDADES BÁSICAS
# ─────────────────────────────────────────────────────────────

def cat_to_num(valor, mapa, defecto=0.5):
    """
    Convierte un valor categórico string o numérico a float.
    Actúa sobre un elemento, nunca sobre una Serie completa
    → no hay TypeError por mezcla de tipos.

    Ejemplos:
      cat_to_num('Low',  STRESS_MAP)  → 2.0
      cat_to_num(7.3,    STRESS_MAP)  → 7.3   (ya es número)
      cat_to_num(None,   STRESS_MAP)  → 0.5   (defecto)
    """
    if valor is None or (isinstance(valor, float) and np.isnan(valor)):
        return defecto
    if isinstance(valor, str):
        return float(mapa.get(valor.lower().strip(), defecto))
    try:
        return float(valor)
    except (ValueError, TypeError):
        return defecto


def normalizar(serie, minimo, maximo):
    """Clipea y escala una columna a [0, 1]."""
    s = pd.to_numeric(serie, errors='coerce').fillna((minimo + maximo) / 2)
    return (s.clip(minimo, maximo) - minimo) / (maximo - minimo)


def escalar_0_100(serie):
    """Lleva cualquier score numérico a rango [0, 100]."""
    s = pd.to_numeric(serie, errors='coerce')
    mn, mx = s.min(), s.max()
    return ((s - mn) / (mx - mn) * 100) if mx > mn else s * 0 + 50.0


# Mapas categórico → numérico
STRESS_MAP   = {'low':2.0, 'mild':3.0, 'moderate':5.5, 'medium':5.5,
                'high':8.0, 'severe':9.0, 'very high':9.5}

EXERCISE_MAP = {'none':1, 'sedentary':1,                 # 1 = sin actividad
                'low':2,  'light':2,                      # 2 = baja
                'moderate':3, 'medium':3,                 # 3 = moderada ← óptimo
                'high':4, 'active':4,                     # 4 = alta
                'very high':5, 'very active':5}           # 5 = muy alta

EXERCISE_NORM = {'none':0.0, 'sedentary':0.05,           # fracción actividad [0-1]
                 'low':0.2,  'light':0.2,                 # para clustering
                 'moderate':0.5, 'medium':0.5,
                 'high':0.8, 'active':0.8,
                 'very high':1.0, 'very active':1.0}


# ─────────────────────────────────────────────────────────────
# PASO 1A — Limpiar Dataset A (Mental Health & Lifestyle)
# ─────────────────────────────────────────────────────────────
# Columnas de entrada (Kaggle):
#   Sleep Hours, Stress Level (string), Exercise Level (string),
#   Screen Time per Day (Hours), Social Interaction Score,
#   Happiness Score, Age, Gender

def limpiar_A(path='mental_health.csv'):
    df = pd.read_csv(path)
    print(f"\n[A] Cargado: {df.shape}")

    # Renombrar a nombres internos
    df = df.rename(columns={
        'Sleep Hours':                 'sleep_hours',
        'Stress Level':                'stress_raw',      # string → convertir
        'Exercise Level':              'exercise_raw',    # string → convertir
        'Screen Time per Day (Hours)': 'screen_time',
        'Social Interaction Score':    'social_score',
        'Happiness Score':             'happiness_score',
        'Age':                         'age',
        'Gender':                      'gender',
    })

    # Convertir stress y exercise de string a numérico
    df['stress_num']   = df['stress_raw'].apply(
                             lambda x: cat_to_num(x, STRESS_MAP, defecto=5.0))
    df['exercise_int'] = df['exercise_raw'].apply(
                             lambda x: cat_to_num(x, EXERCISE_MAP, defecto=3))
    df['exercise_int'] = df['exercise_int'].astype(int)

    # Limpiar columnas numéricas
    df['sleep_hours']  = pd.to_numeric(df['sleep_hours'],  errors='coerce')
    df['screen_time']  = pd.to_numeric(df['screen_time'],  errors='coerce')
    df['social_score'] = pd.to_numeric(df['social_score'], errors='coerce')
    df['happiness_score'] = pd.to_numeric(df['happiness_score'], errors='coerce')
    df['age']          = pd.to_numeric(df['age'],          errors='coerce')

    # Descartar filas con valores esenciales nulos
    df = df.dropna(subset=['sleep_hours', 'happiness_score', 'stress_num'])
    df = df.reset_index(drop=True)
    print(f"[A] Limpio: {df.shape}")
    return df


# ─────────────────────────────────────────────────────────────
# PASO 1B — Limpiar Dataset B (Holistic Health & Lifestyle Score)
# ─────────────────────────────────────────────────────────────
# Columnas de entrada (Kaggle):
#   Sleep_Hours, Stress_Level (float 1-10),
#   Physical_Activity (float 0-100), Nutrition_Score,
#   Mindfulness, Hydration, BMI, Alcohol, Smoking,
#   Overall_Health_Score

def limpiar_B(path='holistic_health.csv'):
    df = pd.read_csv(path)
    print(f"\n[B] Cargado: {df.shape}")

    df = df.rename(columns={
        'Sleep_Hours':          'sleep_hours',
        'Stress_Level':         'stress_num',       # ya es float 1-10
        'Physical_Activity':    'physical_activity',# float 0-100
        'Nutrition_Score':      'nutrition_score',
        'Mindfulness':          'mindfulness',
        'Hydration':            'hydration',
        'BMI':                  'bmi',
        'Alcohol':              'alcohol',
        'Smoking':              'smoking',
        'Overall_Health_Score': 'overall_health_score'
    })

    # Limpiar tipos
    for col in ['sleep_hours','stress_num','physical_activity',
                'nutrition_score','mindfulness','hydration',
                'bmi','alcohol','overall_health_score']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # exercise_intensity en B: derivar desde physical_activity (0-100)
    # 0-20 → 1, 20-40 → 2, 40-60 → 3, 60-80 → 4, 80-100 → 5
    df['exercise_int'] = pd.cut(
        df['physical_activity'].clip(0, 100),
        bins=[0, 20, 40, 60, 80, 100],
        labels=[1, 2, 3, 4, 5]
    ).astype(float).astype('Int64').fillna(3)

    df = df.dropna(subset=['sleep_hours', 'overall_health_score', 'stress_num'])
    df = df.reset_index(drop=True)
    print(f"[B] Limpio: {df.shape}")
    return df


In [14]:

# ─────────────────────────────────────────────────────────────
# PASO 2A — Construir bienestar_mental desde Dataset A
# ─────────────────────────────────────────────────────────────

def construir_mental(dfA):
    """
    happiness_score (escala original del dataset) → bienestar_mental [0, 100].
    Es el target que el modelo aprenderá a predecir.
    """
    dfA = dfA.copy()
    dfA['bienestar_mental'] = escalar_0_100(dfA['happiness_score'])
    return dfA


# ─────────────────────────────────────────────────────────────
# PASO 2B — Construir bienestar_fisico desde Dataset B
# ─────────────────────────────────────────────────────────────

def construir_fisico(dfB):
    """
    overall_health_score → bienestar_fisico [0, 100].
    """
    dfB = dfB.copy()
    dfB['bienestar_fisico'] = escalar_0_100(dfB['overall_health_score'])
    return dfB


# ─────────────────────────────────────────────────────────────
# PASO 3 — Normalizar features comunes para clustering
# ─────────────────────────────────────────────────────────────
# Solo las 3 dimensiones genuinamente presentes en ambos datasets.
# No rellenamos con 0.5 lo que falta: eso introduce ruido artificial.

def features_comunes(dfA, dfB):
    """
    Devuelve dos DataFrames con el espacio [0,1]^3:
    [sleep_norm, stress_norm, exercise_norm]
    """
    fA = pd.DataFrame({
        'sleep'    : normalizar(dfA['sleep_hours'], 3, 12),
        'stress'   : dfA['stress_num'] / 10.0,            # ya es 1-10
        'exercise' : dfA['exercise_raw'].apply(
                         lambda x: cat_to_num(x, EXERCISE_NORM, 0.5)),
    })

    fB = pd.DataFrame({
        'sleep'    : normalizar(dfB['sleep_hours'], 3, 12),
        'stress'   : normalizar(dfB['stress_num'], 1, 10),
        'exercise' : normalizar(dfB['physical_activity'], 0, 100),
    })

    return fA.reset_index(drop=True), fB.reset_index(drop=True)



In [15]:

# ─────────────────────────────────────────────────────────────
# PASO 4 — Merge: clustering + matching + sampleo
# ─────────────────────────────────────────────────────────────

def merge_por_clustering(dfA, dfB, fA, fB, k=12, max_x_cluster=80, umbral=0.40):
    """
    1. KMeans k=12 sobre cada dataset en el espacio 3D común
    2. Emparejar clusters por distancia mínima entre centroides
    3. Por cada par: samplear min(nA, nB, max) filas
    4. Cada fila hereda features de A + features de B

    Resultado: DataFrame con las columnas de ambos datasets
    y las dos etiquetas por fila.
    """
    kmA = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(fA)
    kmB = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(fB)

    # Matriz de distancias entre los 12×12 centroides
    D = cdist(kmA.cluster_centers_, kmB.cluster_centers_)

    # Matching greedy: cada cluster A → el B más cercano no usado
    matches, usados = [], set()
    for cA in range(k):
        for cB in np.argsort(D[cA]):
            if cB not in usados:
                matches.append((cA, int(cB), D[cA, cB]))
                usados.add(cB)
                break

    matches = [(a, b, d) for a, b, d in matches if d < umbral]
    print(f"\n[Merge] Clusters emparejados (d < {umbral}): {len(matches)}/{k}")
    for a, b, d in matches:
        print(f"  A[{a:2d}]({(kmA.labels_==a).sum():4d}) ↔ "
              f"B[{b:2d}]({(kmB.labels_==b).sum():4d})  d={d:.3f}")

    # Sampleo de filas por par de clusters
    filas = []
    for cA, cB, _ in matches:
        idxA = np.where(kmA.labels_ == cA)[0]
        idxB = np.where(kmB.labels_ == cB)[0]
        n    = min(len(idxA), len(idxB), max_x_cluster)
        if n < 5:
            continue
        for iA, iB in zip(rng.choice(idxA, n, replace=False),
                          rng.choice(idxB, n, replace=False)):
            # Combinar fila de A y fila de B en un dict plano
            fila = {
                # De A
                'stress_A'       : dfA.loc[iA, 'stress_num'],
                'sleep_A'        : dfA.loc[iA, 'sleep_hours'],
                'screen_A'       : dfA.loc[iA, 'screen_time'],
                'social_A'       : dfA.loc[iA, 'social_score'],
                'exercise_int_A' : dfA.loc[iA, 'exercise_int'],
                'age'            : dfA.loc[iA, 'age'],
                'gender'         : dfA.loc[iA, 'gender'],
                'bienestar_mental': dfA.loc[iA, 'bienestar_mental'],
                # De B
                'stress_B'         : dfB.loc[iB, 'stress_num'],
                'sleep_B'          : dfB.loc[iB, 'sleep_hours'],
                'exercise_int_B'   : dfB.loc[iB, 'exercise_int'],
                'physical_activity': dfB.loc[iB, 'physical_activity'],
                'nutrition_score'  : dfB.loc[iB, 'nutrition_score'],
                'mindfulness'      : dfB.loc[iB, 'mindfulness'],
                'hydration'        : dfB.loc[iB, 'hydration'],
                'bmi'              : dfB.loc[iB, 'bmi'],
                'alcohol'          : dfB.loc[iB, 'alcohol'],
                'bienestar_fisico' : dfB.loc[iB, 'bienestar_fisico'],
            }
            filas.append(fila)

    df = pd.DataFrame(filas)
    print(f"[Merge] Filas resultantes: {len(df)}")
    return df

In [16]:

# ─────────────────────────────────────────────────────────────
# PASO 5 — Construir features finales desde el merged
# ─────────────────────────────────────────────────────────────
# Aquí se resuelven los campos que la app va a registrar
# y que no estaban directamente en ningún dataset.

def construir_features(df):
    """
    Transforma el dataset merged (con columnas crudas de A y B)
    al schema final de la app.
    """
    df = df.copy()
    out = pd.DataFrame()

    # ── Perfil estático ──────────────────────────────────────
    out['edad']   = pd.to_numeric(df['age'], errors='coerce').fillna(30).astype(int)
    sexo_vec      = df['gender'].apply(
                        lambda g: 1 if isinstance(g, str) and 'm' in g.lower() else 0)
    out['sexo']   = sexo_vec
    # Altura sintética plausible (no está en ningún dataset)
    out['altura'] = sexo_vec.apply(
        lambda s: round(float(np.clip(rng.normal(1.70 if s == 1 else 1.62, 0.07), 1.50, 2.00)), 2))
    bmi           = df['bmi'].clip(16, 45)
    out['bmi']    = bmi.round(1)
    out['peso']   = (bmi * out['altura'] ** 2).round(1)

    # ── Wearable (automático) ────────────────────────────────
    # Sueño: media de A y B (ambos tienen sleep_hours)
    out['h_sueno']  = ((df['sleep_A'] + df['sleep_B']) / 2).clip(3, 12).round(1)

    # Calidad sueño: proxy desde estrés y horas
    stress_medio    = ((df['stress_A'] + df['stress_B']) / 2).clip(1, 10)
    out['c_sueno']  = (5 - (stress_medio / 10) * 2
                       - (df['sleep_A'] < 6).astype(float) * 0.8
                       ).clip(1, 5).round(1)

    # Estrés: media de ambas fuentes → float 1-10 limpio
    out['n_estres'] = stress_medio.round(1)

    # Pantalla (solo en A)
    out['h_movil']  = df['screen_A'].clip(0, 14).round(1)
    out['t_pantalla'] = (df['screen_A'] * 1.1).clip(0, 16).round(1)

    # Pasos y ejercicio: desde physical_activity de B (0-100)
    pa_norm         = (df['physical_activity'] / 100.0).clip(0, 1)
    out['pasos']    = (2000 + pa_norm * 13000).astype(int)

    # Intensidad ejercicio 1-5: combinar A y B
    # A tiene escala categórica; B tiene escala derivada de physical_activity
    # Tomamos el máximo como señal más conservadora (no infraestimar esfuerzo)
    out['i_ejercicio'] = df[['exercise_int_A','exercise_int_B']].max(axis=1).astype(int)
    out['m_ejercicio'] = (pa_norm * 120).astype(int)

    # ── Encuesta diaria ──────────────────────────────────────
    # Nutrición desde nutrition_score de B
    ns_n = (df['nutrition_score'] / df['nutrition_score'].max()).clip(0, 1)
    out['alim_enteros']    = (ns_n * 10).clip(1, 10).round(1)
    out['alim_procesados'] = ((1 - ns_n) * 8).clip(0, 8).round(1)
    out['g_proteina']      = (ns_n * out['peso'] * 2.0).clip(40, 200).round(1)
    out['g_fibra']         = (ns_n * 45).clip(5, 45).round(1)

    # Social solo en A
    out['h_social']     = (df['social_A'] / df['social_A'].max() * 8).clip(0, 8).round(1)

    # Sol: proxy desde actividad física (más activo → más tiempo fuera)
    out['h_sol']        = (pa_norm * 4.5).clip(0, 6).round(1)

    out['l_agua']       = df['hydration'].clip(0.5, 4).round(2)
    out['n_alcohol']    = df['alcohol'].clip(0, 10).round(1)
    out['t_meditacion'] = (df['mindfulness'] * 30).clip(0, 60).round(1)

    # ── Etiquetas ────────────────────────────────────────────
    out['bienestar_fisico']  = df['bienestar_fisico'].round(2)
    out['bienestar_mental']  = df['bienestar_mental'].round(2)

    print(f"\n[Features] Schema final: {list(out.columns)}")
    return out.reset_index(drop=True)



In [17]:


# ─────────────────────────────────────────────────────────────
# PIPELINE COMPLETO
# ─────────────────────────────────────────────────────────────

def run(path_a="/content/Mental_Health_Lifestyle_Dataset.csv", path_b="/content/holistic_health_lifestyle_dataset.csv"):
    print("=" * 50)
    print("Pipeline — Dataset de Bienestar")
    print("=" * 50)

    # 1. Limpiar
    dfA = limpiar_A(path_a)
    dfB = limpiar_B(path_b)

    # 2. Construir etiquetas
    dfA = construir_mental(dfA)
    dfB = construir_fisico(dfB)

    # 3. Normalizar features comunes
    fA, fB = features_comunes(dfA, dfB)

    # 4. Merge por clustering
    merged = merge_por_clustering(dfA, dfB, fA, fB)

    # 5. Features finales
    df = construir_features(merged)
    df_final = pd.concat([df, hist_df])

    df_final.to_csv('wellness_dataset.csv', index=False)
    print(df_final.iloc[0])
    print(f"\n✓ wellness_dataset.csv  ({len(df_final)} filas × {len(df.columns)} columnas)")
    return df


# ─────────────────────────────────────────────────────────────
# DEMO SIN CSVs REALES
# ─────────────────────────────────────────────────────────────

def demo():
    """Genera datos sintéticos imitando los tipos reales de Kaggle."""
    n_a, n_b = 3000, 10000

    dfA = pd.DataFrame({
        'sleep_hours'    : rng.normal(6.8, 1.4, n_a).clip(3, 12),
        'stress_raw'     : rng.choice(['Low','Moderate','High'], n_a),
        'screen_time'    : rng.normal(5.1, 2.0, n_a).clip(0, 14),
        'social_score'   : rng.uniform(1, 10, n_a),
        'happiness_score': rng.normal(6.5, 1.8, n_a).clip(1, 10),
        'exercise_level' : rng.choice(['Low','Moderate','High','Very High'], n_a),
        'age'            : rng.integers(18, 65, n_a),
        'gender'         : rng.choice(['Male','Female'], n_a),
    })

    dfB = pd.DataFrame({
        'sleep_hours'         : rng.normal(7.0, 1.3, n_b).clip(3, 12),
        'stress_raw'          : rng.uniform(1, 10, n_b),
        'physical_activity'   : rng.uniform(20, 90, n_b),
        'nutrition_score'     : rng.uniform(2, 9, n_b),
        'mindfulness'         : rng.uniform(0, 1, n_b),
        'hydration'           : rng.normal(1.8, 0.6, n_b).clip(0.5, 4),
        'bmi'                 : rng.normal(25, 4, n_b).clip(16, 45),
        'alcohol'             : rng.exponential(1.2, n_b).clip(0, 8),
        'smoking'             : rng.choice([0, 1], n_b, p=[0.8, 0.2]),
        'overall_health_score': rng.normal(65, 18, n_b).clip(10, 100),
    })

    # Correlaciones realistas
    dfB['stress_raw'] = (dfB['stress_raw']
                         - 0.39 * (dfB['sleep_hours'] - 7)).clip(1, 10)
    dfB['overall_health_score'] = (
        dfB['overall_health_score']
        + 0.35 * (dfB['physical_activity'] - 55) / 45 * 20).clip(10, 100)

    # Renombrar para que coincidan con limpiar_A / limpiar_B
    dfA = dfA.rename(columns={'exercise_level':'exercise_raw',
                               'stress_raw':'stress_raw'})
    dfB = dfB.rename(columns={'stress_raw':'stress_num',
                               'overall_health_score':'overall_health_score'})
    dfA['exercise_int'] = dfA['exercise_raw'].apply(
                              lambda x: cat_to_num(x, EXERCISE_MAP, 3))
    dfA['exercise_int'] = dfA['exercise_int'].astype(int)
    dfA['stress_num']   = dfA['stress_raw'].apply(
                              lambda x: cat_to_num(x, STRESS_MAP, 5.0))

    dfB['exercise_int'] = pd.cut(
        dfB['physical_activity'].clip(0, 100),
        bins=[0,20,40,60,80,100], labels=[1,2,3,4,5]
    ).astype(float).astype('Int64').fillna(3)

    dfA['bienestar_mental'] = escalar_0_100(dfA['happiness_score'])
    dfB['bienestar_fisico'] = escalar_0_100(dfB['overall_health_score'])

    fA = pd.DataFrame({
        'sleep'   : normalizar(dfA['sleep_hours'], 3, 12),
        'stress'  : dfA['stress_num'] / 10.0,
        'exercise': dfA['exercise_raw'].apply(
                        lambda x: cat_to_num(x, EXERCISE_NORM, 0.5)),
    })
    fB = pd.DataFrame({
        'sleep'   : normalizar(dfB['sleep_hours'], 3, 12),
        'stress'  : normalizar(dfB['stress_num'], 1, 10),
        'exercise': normalizar(dfB['physical_activity'], 0, 100),
    })

    print("[DEMO] Tipos: stress A=string, stress B=float, physical_activity 0-100")
    merged = merge_por_clustering(dfA, dfB, fA, fB)
    df     = construir_features(merged)
    df_final = pd.concat([df, hist_df], ignore_index=True)
    df_final.to_csv('wellness_dataset_demo.csv', index=False)
    print(f"\n✓ wellness_dataset_demo.csv  ({len(df_final)} filas)")
    print(f"\nMuestra:")
    print(df_final[['edad','h_sueno','n_estres','i_ejercicio',
              'm_ejercicio','bienestar_fisico','bienestar_mental']].head(5).to_string())


if __name__ == '__main__':
    import sys
    demo() if '--demo' in sys.argv else run()


Pipeline — Dataset de Bienestar

[A] Cargado: (3000, 12)
[A] Limpio: (3000, 14)

[B] Cargado: (10000, 11)
[B] Limpio: (10000, 12)

[Merge] Clusters emparejados (d < 0.4): 11/12
  A[ 0]( 249) ↔ B[ 8]( 761)  d=0.124
  A[ 1]( 189) ↔ B[10]( 723)  d=0.125
  A[ 2]( 326) ↔ B[ 9]( 784)  d=0.067
  A[ 3]( 254) ↔ B[ 1]( 746)  d=0.095
  A[ 4]( 356) ↔ B[ 2]( 710)  d=0.086
  A[ 5]( 204) ↔ B[ 5]( 885)  d=0.129
  A[ 6]( 237) ↔ B[ 3]( 707)  d=0.194
  A[ 7]( 324) ↔ B[ 6](1001)  d=0.235
  A[ 8]( 243) ↔ B[ 7]( 875)  d=0.218
  A[ 9]( 194) ↔ B[11]( 821)  d=0.175
  A[10]( 190) ↔ B[ 0]( 729)  d=0.173
[Merge] Filas resultantes: 880

[Features] Schema final: ['edad', 'sexo', 'altura', 'bmi', 'peso', 'h_sueno', 'c_sueno', 'n_estres', 'h_movil', 't_pantalla', 'pasos', 'i_ejercicio', 'm_ejercicio', 'alim_enteros', 'alim_procesados', 'g_proteina', 'g_fibra', 'h_social', 'h_sol', 'l_agua', 'n_alcohol', 't_meditacion', 'bienestar_fisico', 'bienestar_mental']
edad                  39.00
sexo                   1.00
alt

In [18]:
git remote add origin https://github.com/maybepueblo/wellness_prediction_app.git
git branch -M main
git push -u origin main

SyntaxError: invalid syntax (806531521.py, line 1)